# In-Class 9: Comparing Tree-Based Feature Importance Methods
Marine-science application: predicting abalone age

In this activity, we will fit one random-forest regression model and explain it in three different ways:

1. Impurity-based importance (mean decrease in impurity, or MDI)
2. Permutation importance measured on held-out test data
3. SHAP values for global and individual prediction explanations

Using the same fitted model and the same predictors keeps the comparison fair. The goal is not to identify one universally “correct” importance method. Instead, we will learn what question each method answers and why their rankings can differ.
Learning objectives

By the end of class, you should be able to:

- distinguish model-specific importance from performance-based importance;
- calculate and visualize MDI, permutation importance, and SHAP importance;
- explain why correlated predictors can divide or hide importance;
- distinguish global explanations from local explanations;
- identify common ways feature-importance plots can be overinterpreted.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

## Load and inspect the abalone data

In [ ]:
abalone = pd.read_csv(
    "https://raw.githubusercontent.com/dblaskey/ML_Course_Code/main/Data/abalone.csv",
    header=None,
    names=["sex", "length", "diameter", "height", "whole_weight", "shucked_weight", "viscera_weight", "shell_weight", "rings"]
)

print("Shape:", abalone.shape)
display(abalone.head())
display(abalone.describe(include="all").T)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

numeric_data = abalone.select_dtypes(include="number")
corr = numeric_data.corr()

# Hide the duplicated upper triangle
upper_triangle = np.triu(
    np.ones_like(corr, dtype=bool),
    k=1
)

masked_corr = np.ma.array(
    corr.to_numpy(),
    mask=upper_triangle
)

# Use Seaborn's polished diverging color palette
cmap = sns.color_palette("vlag", as_cmap=True).copy()
cmap.set_bad("white")

fig, ax = plt.subplots(figsize=(10, 8), constrained_layout=True)

image = ax.imshow(
    masked_corr,
    cmap=cmap,
    vmin=-1,
    vmax=1,
    aspect="equal",
)

# Add annotations to the visible cells
for row in range(corr.shape[0]):
    for col in range(corr.shape[1]):

        if not upper_triangle[row, col]:
            value = corr.iloc[row, col]

            # White text over dark cells; dark text over light cells
            text_color = "white" if abs(value) > 0.55 else "#333333"

            ax.text(
                col,
                row,
                f"{value:.2f}",
                ha="center",
                va="center",
                color=text_color,
                fontsize=9,
                fontweight="medium",
            )

# Axis positions and labels
ax.set_xticks(np.arange(corr.shape[1]))
ax.set_yticks(np.arange(corr.shape[0]))

ax.set_xticklabels(corr.columns)
ax.set_yticklabels(corr.index)

plt.setp(
    ax.get_xticklabels(),
    rotation=45,
    ha="right",
    rotation_mode="anchor",
)

# Add subtle white borders between cells
ax.set_xticks(
    np.arange(-0.5, corr.shape[1], 1),
    minor=True,
)
ax.set_yticks(
    np.arange(-0.5, corr.shape[0], 1),
    minor=True,
)

ax.grid(
    which="minor",
    color="white",
    linewidth=1.5,
)

ax.tick_params(which="minor", bottom=False, left=False)
ax.tick_params(axis="both", length=0, labelsize=10)

# Remove the outside frame
for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_xlim(-0.5, corr.shape[1] - 0.5)
ax.set_ylim(corr.shape[0] - 0.5, -0.5)

colorbar = fig.colorbar(
    image,
    ax=ax,
    shrink=0.78,
    pad=0.03,
)

colorbar.set_label(
    "Pearson correlation",
    rotation=270,
    labelpad=18,
)

ax.set_title(
    "Correlations Among Abalone Measurements",
    fontsize=15,
    pad=18,
)

plt.show()

## Split the data

In [ ]:
# One-hot encode 'Sex' and separate features
X = pd.get_dummies(abalone.drop("rings", axis=1), columns=["sex"], drop_first=True)
y = abalone["rings"]

# Feature names
feature_names = X.columns

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## Fit and evaluate the model

In [ ]:
forest = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=3,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42,
)
forest.fit(X_train, y_train)

In [ ]:
forest_pred = forest.predict(X_test)
print("RMSE: ", mean_squared_error(y_test, forest_pred) ** 0.5)
print("MAE: ", mean_absolute_error(y_test, forest_pred))
print("R2: ", r2_score(y_test, forest_pred))

## Method 1: impurity-based importance

For a regression forest, each split is selected to reduce squared-error impurity. Scikit-learn accumulates the weighted impurity reductions attributed to each feature and averages them across trees. The normalized results sum to 1.

This is often called **mean decrease in impurity (MDI)**.

### What question does it answer?

> How much did the fitted forest use this feature to reduce training impurity across all of its splits?

### Strengths

- essentially free after model fitting;
- fast and easy to calculate;
- summarizes how the fitted forest constructed its trees.

### Limitations

- calculated from training splits rather than held-out predictive performance;
- can favor continuous or high-cardinality predictors because they provide more possible split points;
- correlated predictors can share or steal importance;
- does not show whether a feature pushes predictions higher or lower;
- global importance does not explain any single observation.

In [ ]:
mdi_importance = (
    pd.DataFrame(
        {
            "feature": feature_names,
            "mdi": forest.feature_importances_,
        }
    )
    .sort_values("mdi", ascending=False)
    .reset_index(drop=True)
)

display(mdi_importance.round(4))

plt.figure(figsize=(8, 5.5))
sns.barplot(data=mdi_importance, x="mdi", y="feature", color="steelblue")
plt.xlabel("Mean decrease in impurity")
plt.ylabel("")
plt.title("Random-forest impurity-based importance")
plt.tight_layout()
plt.show()

## Method 2: permutation importance

Permutation importance measures how much model performance worsens after one predictor is shuffled. Shuffling breaks that predictor's relationship with the outcome while leaving the fitted model unchanged.

We calculate it on the **test set** and use RMSE as the scoring metric.

For this scoring setup:

$$
\text{Permutation importance}_j
=
\text{RMSE after shuffling feature }j
-
\text{original RMSE}
$$

A large positive value means shuffling the predictor increased test error substantially.

### What question does it answer?

> How much does held-out predictive performance depend on this feature?

### Strengths

- evaluated with a chosen performance metric;
- can be calculated on held-out data;
- model-agnostic;
- repeated permutations provide a measure of variability.

### Limitations

- correlated predictors can substitute for one another, making each look less important;
- shuffling may create unrealistic combinations of environmental measurements;
- results depend on the evaluation set and scoring metric;
- slower than built-in MDI.

In [ ]:
permutation_result = permutation_importance(
    forest,
    X_test,
    y_test,
    scoring="neg_root_mean_squared_error",
    n_repeats=20,
    random_state=42,
    n_jobs=-1,
)

permutation_df = (
    pd.DataFrame(
        {
            "feature": feature_names,
            "permutation_mean": permutation_result.importances_mean,
            "permutation_sd": permutation_result.importances_std,
        }
    )
    .sort_values("permutation_mean", ascending=False)
    .reset_index(drop=True)
)

display(permutation_df.round(4))

In [ ]:
plot_df = permutation_df.sort_values("permutation_mean", ascending=True)

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.barh(
    plot_df["feature"],
    plot_df["permutation_mean"],
    xerr=plot_df["permutation_sd"],
    color="darkorange",
    alpha=0.85,
    capsize=3,
)
ax.axvline(0, color="black", linewidth=1)
ax.set_xlabel("Increase in test RMSE after permutation")
ax.set_ylabel("")
ax.set_title("Permutation importance on held-out data")
plt.tight_layout()
plt.show()

### Question 1

1. Which features differ most between the MDI and permutation rankings?
2. What does an importance close to zero mean?
4. Could an importance be negative? What would that suggest?

## Method 3: SHAP values

SHAP assigns each feature a contribution to a prediction relative to a baseline prediction. For one observation:

$$
\widehat{y}_i
=
\text{baseline}
+
\sum_{j=1}^{p}\phi_{ij}
$$

where $\phi_{ij}$ is the SHAP contribution of feature $j$ for observation $i$.

- Positive SHAP value: pushes the prediction above the baseline.
- Negative SHAP value: pushes the prediction below the baseline.
- Large absolute SHAP value: strong contribution for that observation.

Global SHAP importance is usually calculated as the mean absolute SHAP value:

$$
I_j=\frac{1}{n}\sum_{i=1}^{n}|\phi_{ij}|
$$

For this regression model, the values are measured in the target's units: **rings**.

### What question does it answer?

> How much does each feature contribute to individual predictions, relative to the model's baseline output?

We use a reproducible subset of the test data to keep the classroom calculation fast.

In [ ]:
N_SHAP = min(300, len(X_test))
X_shap = X_test.sample(n=N_SHAP, random_state=42)

explainer = shap.TreeExplainer(forest)
shap_values = explainer(X_shap)

print("SHAP matrix shape:", shap_values.values.shape)
print("Expected features:", (N_SHAP, X_test.shape[1]))

### Global SHAP bar plot

The bar length is the mean absolute SHAP value. It describes average contribution magnitude but not direction.

In [ ]:
shap.plots.bar(shap_values, max_display=len(feature_names), show=False)
plt.title("Global SHAP importance: mean absolute contribution")
plt.tight_layout()
plt.show()

### SHAP beeswarm plot

Each point is one observation:

- horizontal position = SHAP contribution;
- color = observed feature value;
- vertical order = mean absolute SHAP importance.

Unlike MDI and permutation importance, the beeswarm can show both the magnitude and direction of contributions.

In [ ]:
shap.plots.beeswarm(shap_values, max_display=len(feature_names), show=False)
plt.title("SHAP contributions across test observations")
plt.tight_layout()
plt.show()

### Question 2

1. Which high feature values generally push predicted rings upward?
2. Are any relationships non-monotonic—that is, do both high and low feature values have mixed effects?

## Compare all three global rankings

The raw importance values are not directly comparable because they use different units:

- MDI is a normalized fraction of accumulated impurity reduction;
- permutation importance is an increase in RMSE;
- SHAP importance is a mean absolute contribution measured in rings.

For visualization only, we convert each column to a proportion of its own total. This compares relative emphasis, not absolute magnitude.

In [ ]:
shap_importance = pd.DataFrame(
    {
        "feature": X_shap.columns,
        "shap_mean_abs": np.abs(shap_values.values).mean(axis=0),
    }
)

importance_comparison = (
    mdi_importance
    .merge(permutation_df, on="feature")
    .merge(shap_importance, on="feature")
)

for column in ["mdi", "permutation_mean", "shap_mean_abs"]:
    denominator = importance_comparison[column].clip(lower=0).sum()
    importance_comparison[f"{column}_share"] = (
        importance_comparison[column].clip(lower=0) / denominator
    )

importance_comparison["mdi_rank"] = importance_comparison["mdi"].rank(
    ascending=False, method="min"
)
importance_comparison["permutation_rank"] = importance_comparison[
    "permutation_mean"
].rank(ascending=False, method="min")
importance_comparison["shap_rank"] = importance_comparison["shap_mean_abs"].rank(
    ascending=False, method="min"
)

display(
    importance_comparison[
        [
            "feature",
            "mdi",
            "permutation_mean",
            "shap_mean_abs",
            "mdi_rank",
            "permutation_rank",
            "shap_rank",
        ]
    ]
    .sort_values("shap_rank")
    .round(4)
)

In [ ]:
comparison_long = importance_comparison.melt(
    id_vars="feature",
    value_vars=[
        "mdi_share",
        "permutation_mean_share",
        "shap_mean_abs_share",
    ],
    var_name="method",
    value_name="relative_importance",
)

method_labels = {
    "mdi_share": "Impurity (MDI)",
    "permutation_mean_share": "Permutation",
    "shap_mean_abs_share": "Mean |SHAP|",
}
comparison_long["method"] = comparison_long["method"].map(method_labels)

feature_order = (
    importance_comparison.sort_values("shap_mean_abs", ascending=False)["feature"]
    .tolist()
)

plt.figure(figsize=(11, 6))
sns.barplot(
    data=comparison_long,
    x="feature",
    y="relative_importance",
    hue="method",
    order=feature_order,
)
plt.xticks(rotation=45, ha="right")
plt.xlabel("")
plt.ylabel("Share of importance within method")
plt.title("Relative feature importance across three methods")
plt.legend(title="")
plt.tight_layout()
plt.show()

In [ ]:
rank_columns = ["mdi_rank", "permutation_rank", "shap_rank"]
rank_agreement = importance_comparison[rank_columns].corr(method="spearman")
method_names = ["Impurity (MDI)", "Permutation", "Mean |SHAP|"]

# Give the table readable labels
rank_agreement.index = method_names
rank_agreement.columns = method_names

display(rank_agreement.round(3))

# Use Matplotlib directly because Seaborn is not displaying
# all annotations correctly in this environment.
fig, ax = plt.subplots(figsize=(6.5, 5.5),constrained_layout=True)
cmap = sns.color_palette("vlag", as_cmap=True)

image = ax.imshow(
    rank_agreement.to_numpy(),
    cmap=cmap,
    vmin=-1,
    vmax=1,
    aspect="equal",
)

# Manually annotate every cell
for row in range(rank_agreement.shape[0]):
    for col in range(rank_agreement.shape[1]):
        value = rank_agreement.iloc[row, col]

        text_color = ("white" if abs(value) > 0.60 else "#333333")
        ax.text(col, row, f"{value:.2f}", ha="center", va="center", color=text_color, fontsize=12, fontweight="medium")

ax.set_xticks(np.arange(len(method_names)))
ax.set_yticks(np.arange(len(method_names)))
ax.set_xticklabels(method_names, rotation=30, ha="right")
ax.set_yticklabels(method_names)

# Add white borders between cells
ax.set_xticks( np.arange(-0.5, len(method_names), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(method_names), 1), minor=True)
ax.grid(which="minor", color="white", linewidth=2)
ax.tick_params(which="minor", bottom=False, left=False)
ax.tick_params(axis="both", length=0)

for spine in ax.spines.values():
    spine.set_visible(False)

colorbar = fig.colorbar(image, ax=ax, shrink=0.80, pad=0.04)
colorbar.set_label("Spearman rank correlation", rotation=270, labelpad=18)

ax.set_title("Agreement Among Feature-Importance Rankings", fontsize=14, pad=15)
plt.show()

## Local SHAP explanation

Global importance averages over observations. SHAP can also explain one prediction. We select the test observation with the largest absolute prediction error among the observations included in the SHAP calculation.

In [ ]:
shap_predictions = forest.predict(X_shap)
shap_observed = y_test.loc[X_shap.index].to_numpy()
absolute_errors = np.abs(shap_observed - shap_predictions)
local_position = int(np.argmax(absolute_errors))

print("Test-row index:", X_shap.index[local_position])
print("Observed rings:", shap_observed[local_position])
print("Predicted rings:", round(shap_predictions[local_position], 2))
print("Absolute error:", round(absolute_errors[local_position], 2))

shap.plots.waterfall(
    shap_values[local_position],
    max_display=len(feature_names),
    show=False,
)
plt.title("Local explanation for a large-error test observation")
plt.tight_layout()
plt.show()

## Visualize a learned feature relationship

The SHAP scatter plot shows how a feature's observed value relates to its contribution. We automatically select the feature with the largest mean absolute SHAP value. Vertical variation at the same feature value can indicate interactions or dependence on other predictors.

In [ ]:
top_shap_feature = shap_importance.nlargest(1, "shap_mean_abs")["feature"].iloc[0]
print("Top SHAP feature:", top_shap_feature)

shap.plots.scatter(
    shap_values[:, top_shap_feature],
    color=shap_values,
    show=False,
)
plt.title(f"SHAP relationship for {top_shap_feature}")
plt.tight_layout()
plt.show()

## Experiment with correlated predictors

In [ ]:
# ------------------------------------------------------------
# Correlated-feature experiment:
# Remove shell_weight and examine whole_weight
# ------------------------------------------------------------

feature_to_drop = "shell_weight"
feature_to_track = "whole_weight"

X_train_reduced = X_train.drop(columns=feature_to_drop)
X_test_reduced = X_test.drop(columns=feature_to_drop)

# Fit an otherwise identical random forest
reduced_forest = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=3,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42,
)

reduced_forest.fit(X_train_reduced, y_train)

reduced_predictions = reduced_forest.predict(X_test_reduced)

In [ ]:
reduced_mdi = (
    pd.DataFrame({
        "feature": X_train_reduced.columns,
        "mdi": reduced_forest.feature_importances_,
    })
    .sort_values("mdi", ascending=False)
    .reset_index(drop=True)
)

reduced_mdi["mdi_rank"] = (
    reduced_mdi["mdi"]
    .rank(ascending=False, method="min")
)

display(reduced_mdi.round(4))

In [ ]:
reduced_permutation_result = permutation_importance(
    reduced_forest,
    X_test_reduced,
    y_test,
    scoring="neg_root_mean_squared_error",
    n_repeats=20,
    random_state=42,
    n_jobs=-1,
)

reduced_permutation = (
    pd.DataFrame({
        "feature": X_test_reduced.columns,
        "permutation_mean":
            reduced_permutation_result.importances_mean,
        "permutation_sd":
            reduced_permutation_result.importances_std,
    })
    .sort_values("permutation_mean", ascending=False)
    .reset_index(drop=True)
)

reduced_permutation["permutation_rank"] = (
    reduced_permutation["permutation_mean"]
    .rank(ascending=False, method="min")
)

display(reduced_permutation.round(4))

In [ ]:
X_shap_reduced = X_shap.drop(columns=feature_to_drop)

reduced_explainer = shap.TreeExplainer(reduced_forest)
reduced_shap_values = reduced_explainer(X_shap_reduced)

reduced_shap_importance = (
    pd.DataFrame({
        "feature": X_shap_reduced.columns,
        "shap_mean_abs": np.abs(
            reduced_shap_values.values
        ).mean(axis=0),
    })
    .sort_values("shap_mean_abs", ascending=False)
    .reset_index(drop=True)
)

reduced_shap_importance["shap_rank"] = (
    reduced_shap_importance["shap_mean_abs"]
    .rank(ascending=False, method="min")
)

display(reduced_shap_importance.round(4))

In [ ]:
# Combine the three reduced-model importance tables
reduced_importance_comparison = (
    reduced_mdi
    .merge(
        reduced_permutation,
        on="feature",
    )
    .merge(
        reduced_shap_importance,
        on="feature",
    )
)

# Extract whole_weight from the original model
original_whole_weight = (
    importance_comparison.loc[
        importance_comparison["feature"]
        == feature_to_track
    ]
    .iloc[0]
)

# Extract whole_weight from the model without shell_weight
reduced_whole_weight = (
    reduced_importance_comparison.loc[
        reduced_importance_comparison["feature"]
        == feature_to_track
    ]
    .iloc[0]
)

whole_weight_comparison = pd.DataFrame([
    {
        "model": "All predictors",
        "mdi": original_whole_weight["mdi"],
        "mdi_rank": original_whole_weight["mdi_rank"],
        "permutation_mean":
            original_whole_weight["permutation_mean"],
        "permutation_rank":
            original_whole_weight["permutation_rank"],
        "shap_mean_abs":
            original_whole_weight["shap_mean_abs"],
        "shap_rank":
            original_whole_weight["shap_rank"],
    },
    {
        "model": "Without shell_weight",
        "mdi": reduced_whole_weight["mdi"],
        "mdi_rank": reduced_whole_weight["mdi_rank"],
        "permutation_mean":
            reduced_whole_weight["permutation_mean"],
        "permutation_rank":
            reduced_whole_weight["permutation_rank"],
        "shap_mean_abs":
            reduced_whole_weight["shap_mean_abs"],
        "shap_rank":
            reduced_whole_weight["shap_rank"],
    },
])

display(whole_weight_comparison.round(4))

In [ ]:
performance_comparison = pd.DataFrame([
    {
        "model": "All predictors",
        "RMSE": mean_squared_error(
            y_test,
            forest_pred,
        ) ** 0.5,
        "MAE": mean_absolute_error(
            y_test,
            forest_pred,
        ),
        "R2": r2_score(
            y_test,
            forest_pred,
        ),
    },
    {
        "model": "Without shell_weight",
        "RMSE": mean_squared_error(
            y_test,
            reduced_predictions,
        ) ** 0.5,
        "MAE": mean_absolute_error(
            y_test,
            reduced_predictions,
        ),
        "R2": r2_score(
            y_test,
            reduced_predictions,
        ),
    },
])

display(performance_comparison.round(4))

In [ ]:
importance_columns = ["mdi", "permutation_mean", "shap_mean_abs"]
plot_titles = ["Impurity Importance", "Permutation Importance", "Mean Absolute SHAP"]
y_labels = ["MDI", "Increase in test RMSE", "Mean |SHAP| (rings)"]
colors = ["steelblue", "darkorange"]

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

for ax, column, title, ylabel in zip(
    axes,
    importance_columns,
    plot_titles,
    y_labels,
):
    ax.bar(
        whole_weight_comparison["model"],
        whole_weight_comparison[column],
        color=colors,
        width=0.65,
    )

    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=20)

    # Add values above bars
    for position, value in enumerate(whole_weight_comparison[column]):
        ax.text(position, value, f"{value:.3f}", ha="center", va="bottom", fontsize=9)

fig.suptitle("Importance of Whole Weight Before and After Removing Shell Weight", fontsize=14)

plt.tight_layout()
plt.show()

## Summary: choosing an interpretation method

| Method | Primary question | Data used here | Units | Local explanations? | Major caution |
|---|---|---|---|---|---|
| Impurity/MDI | How did the forest reduce training impurity? | Training splits | Proportion summing to 1 | No | Split opportunities and correlated predictors can bias rankings |
| Permutation | How much does test performance depend on a predictor? | Test data | Increase in RMSE | No | Correlation and unrealistic shuffled combinations |
| SHAP | How does each feature contribute to predictions relative to a baseline? | Test subset | Rings | Yes | Background assumptions, dependence, and computational cost |

## Final Questions

1. Did dropping one correlated feature substantially reduce test performance? Did it redistribute importance to another feature? What does this demonstrate about interpreting importance as an intrinsic property of a variable?
2. What additional information does a SHAP beeswarm provide beyond a global SHAP bar plot?